# Part 2, Topic 1, Lab A: Instruction Power Differences (HARDWARE)

---
**THIS IS NOT THE COMPLETE TUTORIAL - see file with `(MAIN)` in the name.**

---

First you'll need to select which hardware setup you have. You'll need to select a `SCOPETYPE`, a `PLATFORM`, and a `CRYPTO_TARGET`. `SCOPETYPE` can either be `'OPENADC'` for the CWLite/CW1200 or `'CWNANO'` for the CWNano. `PLATFORM` is the target device, with `'CWLITEARM'`/`'CW308_STM32F3'` being the best supported option, followed by `'CWLITEXMEGA'`/`'CW308_XMEGA'`, then by `'CWNANO'`. For example:

```python
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWLITEARM'
```

In [ ]:
SCOPETYPE = 'CWNANO'
PLATFORM = 'CWNANO'

## Creating a New Directory

The following bash block will create a new directory and copy the `simpleserial-base` code into it:

In [ ]:
%%bash
cd ../../../firmware/mcu
mkdir -p simpleserial-base-lab2 && cp -r simpleserial-base/* $_
cd simpleserial-base-lab2

## Building Firmware

Rerun this every time you make changes to your code:

In [ ]:
%%bash -s "$PLATFORM"
cd ../../../firmware/mcu/simpleserial-base-lab2
make PLATFORM=$1 CRYPTO_TARGET=NONE -j

## Connecting to Hardware

Let's setup the ChipWhisperer

In [ ]:
import chipwhisperer as cw
try:
    if not scope.connectStatus:
        scope.con()
except NameError:
    scope = cw.scope()

try:
    target = cw.target(scope)
except IOError:
    print("INFO: Caught exception on reconnecting to target - attempting to reconnect to scope first.")
    print("INFO: This is a work-around when USB has died without Python knowing. Ignore errors above this line.")
    scope = cw.scope()
    target = cw.target(scope)

print("INFO: Found ChipWhisperer😍")

if "STM" in PLATFORM or PLATFORM == "CWLITEARM" or PLATFORM == "CWNANO":
    prog = cw.programmers.STM32FProgrammer
else:
    prog = None
    
import time
time.sleep(0.05)
scope.default_setup()
def reset_target(scope):  
    scope.io.nrst = 'low'
    time.sleep(0.05)
    scope.io.nrst = 'high_z'
    time.sleep(0.05)

## Uploading Firmware

The following code will upload firmware to the target. Run it after you've built the firmware:

In [ ]:
cw.program_target(scope, prog, "../../../firmware/mcu/simpleserial-base-lab2/simpleserial-base-{}.hex".format(PLATFORM))

## Capturing a Power Trace

The following is a short function to capture a power trace. We're using `cw.capture_trace()`, which is a short function to replace what we did last time. Check our `readthedocs` page for API documentation: https://chipwhisperer.readthedocs.io/

In [ ]:
def capture_trace(_ignored=None):
    scope.arm()
    
    ktp = cw.ktp.Basic()
    key, text = ktp.next()

    target.simpleserial_write('p', text)
    ret = scope.capture()

    if ret:
        raise RuntimeError("Capture timeout")
    return scope.get_last_trace()

In [ ]:
wave = capture_trace()
print("OK to continue!")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(16,8))
plt.plot(wave)
plt.title("Baseline Trace")
plt.xlabel("Sample")
plt.ylabel("Amplitude")
plt.grid(True)
plt.show()

## Defining the Helper Functions

The next code cell defines helper functions that automate the rest of the lab.

These functions can be used to:
- replace the `get_pt()` function inside the C source file
- restore the original source code later
- rebuild the firmware after each change
- flash the new `.hex` file to the target board
- capture a trace with the updated firmware
- plot the resulting waveform

This automation replaces the manual edit-build-flash cycle that would normally be required.

In [ ]:
from pathlib import Path
import re
import subprocess

FW_FILE = Path("../../../firmware/mcu/simpleserial-base-lab2/simpleserial-base.c")
print("Using firmware source file:")
print(FW_FILE.resolve())

ORIGINAL_SOURCE = FW_FILE.read_text()
print("Original source saved in notebook memory.")

def replace_get_pt(new_function_code: str):
    src = FW_FILE.read_text()
    pattern = r"uint8_t\s+get_pt\s*\(\s*uint8_t\*\s*pt\s*,\s*uint8_t\s+len\s*\)\s*\{.*?\n\}"
    if not re.search(pattern, src, flags=re.DOTALL):
        raise RuntimeError("Could not find get_pt() in simpleserial-base.c")

    updated = re.sub(pattern, new_function_code.strip(), src, flags=re.DOTALL)
    FW_FILE.write_text(updated)
    print("Updated get_pt() in simpleserial-base.c")

def restore_original_source():
    FW_FILE.write_text(ORIGINAL_SOURCE)
    print("Restored original source code.")

def build_firmware():
    subprocess.run(
        ["make", f"PLATFORM={PLATFORM}", "CRYPTO_TARGET=NONE", "-j"],
        cwd=FW_FILE.parent,
        check=True
    )
    print("Build complete.")

def flash_firmware():
    hex_path = "../../../firmware/mcu/simpleserial-base-lab2/simpleserial-base-{}.hex".format(PLATFORM)
    cw.program_target(scope, prog, hex_path)
    print("Flashed:", hex_path)

def capture_named_trace(name):
    trace = capture_trace()
    print(f"Captured: {name}")
    return trace
    
def plot_single_trace(trace, title):
    plt.figure(figsize=(16, 8))
    plt.plot(trace)
    plt.title(title)
    plt.xlabel("Sample")
    plt.ylabel("Amplitude")
    plt.grid(True)
    plt.show()

In [ ]:
GET_PT_ORIGINAL = r"""
uint8_t get_pt(uint8_t* pt, uint8_t len)
{
	/**********************************
	* Start user-specific code here. */
	trigger_high();

	//16 hex bytes held in 'pt' were sent
	//from the computer. Store your response
	//back into 'pt', which will send 16 bytes
	//back to computer. Can ignore of course if
	//not needed

	trigger_low();
	/* End user-specific code here. *
	********************************/
	simpleserial_put('r', 16, pt);
	return 0x00;
}
"""

GET_PT_EMPTY = r"""
uint8_t get_pt(uint8_t* pt, uint8_t len)
{
	/**********************************
	* Start user-specific code here. */
	trigger_high();

	trigger_low();
	/* End user-specific code here. *
	********************************/
	return 0x00;
}
"""

GET_PT_INSTR = r"""
uint8_t get_pt(uint8_t* pt, uint8_t len)
{
	/**********************************
	* Start user-specific code here. */
	trigger_high();

	volatile long int A = 0x2BAA;

	for(int i = 0; i < 1000;i++){
        A += 2;
    }

	trigger_low();
	/* End user-specific code here. *
	********************************/
	return 0x00;
}
"""

## Flashing and plotting the Original Firmware

This is the same trace you used previously, and it is being done again here to set up a baseline.

### What does the original Firmware do?
1. **trigger_high()**
- Sets the trigger signal high.
- This signal is connected to the ChipWhisperer scope.
- The scope uses this signal to know when the interesting part of the code begins.

2. **trigger_low()**
- Sets the trigger signal low again.
- This marks the end of the region of interest.

3. **simpleserial_put('r', 16, pt)**
- Sends 16 bytes back to the host using the SimpleSerial protocol.
- This causes UART communication, which creates additional activity in the power trace.

4. **return 0x00**
- Returns a status value to the SimpleSerial handler.

### Why this matters for power analysis

The power trace we capture reflects everything the microcontroller is doing during execution, including:

- changing GPIO pins (trigger)
- CPU instructions
- memory access
- serial communication

Because the original firmware still sends data back over SimpleSerial, the trace will contain activity related to serial transmission. This makes the waveform more complex and harder to interpret.

### Goal of the experiment

In this lab, we will modify the firmware to create simpler and more controlled so that we can clearly see how instructions affect the power trace.

We will compare three cases:

1. Original firmware (UART communication)
2. Empty firmware (almost no work done)
3. Firmware with repeated instructions

In [ ]:
replace_get_pt(GET_PT_ORIGINAL)
build_firmware()
flash_firmware()

wave_original = capture_named_trace("Original Trace")
plot_single_trace(wave_original, "Original Trace")

In [ ]:
replace_get_pt(GET_PT_EMPTY)
build_firmware()
flash_firmware()

wave_empty = capture_named_trace("Empty Trace")
plot_single_trace(wave_empty, "Empty Trace")

## Capturing the Repeated-Instruction Trace

The next code cell modifies `get_pt()` again so that it repeatedly performs the same arithmetic operation.

This is done to make the processor execute a visible pattern of similar instructions while the trace is being captured.

After rebuilding and reflashing the firmware, the new waveform should show more structure than the empty trace, making it easier to identify where those repeated instructions occur.

It is important to mention that the visualization of different behavior can be hard depending on how many samples are being plotted at once. You might want to change the plotting functions to achieve better visualization depending on your goal.

In [ ]:
replace_get_pt(GET_PT_INSTR)
build_firmware()
flash_firmware()

wave_instr = capture_named_trace("Repeated-Instruction Trace")
plot_single_trace(wave_instr, "Repeated-Instruction Trace")

In [ ]:
plt.figure(figsize=(16, 8))
plt.plot(wave_original, label="Original")
plt.plot(wave_empty, label="Empty")
plt.plot(wave_instr, label="Repeated Instructions")
plt.title("Comparison of Original, Empty, and Repeated-Instruction Traces")
plt.xlabel("Sample")
plt.ylabel("Amplitude")
plt.legend()
plt.grid(True)
plt.show()

## Zoomed Plotting

You are encouraged to change the `start` and `end` values to change the range of traces being plotted in the graph (the range is 0-5000).

In [ ]:
start = 0
end = 500
plt.figure(figsize=(16,8))
plt.plot(wave_original[start:end], label="Original")
plt.plot(wave_empty[start:end], label="Empty")
plt.plot(wave_instr[start:end], label="Repeated")
plt.legend()
plt.grid(True)
plt.title("Zoomed trace")
plt.show()

## (Optional) Restoring Original Firmware

The following cell returns the state of the firmware code to the original state, which makes it easier to reproduce the experiment in the future.

In [ ]:
restore_original_source()
build_firmware()
flash_firmware()
print("Original firmware restored and reflashed")